# 204.1. Calibration frames

For the Rubin Science Platform at data.lsst.cloud. <br>
Data Release: Data Preview 2 <br>
Container Size: large <br>
LSST Science Pipelines version: r29.2.0 <br>
Last verified to run: 2026-04-20 <br>
Repository: <a href="https://github.com/lsst/tutorial-notebooks">github.com/lsst/tutorial-notebooks</a> <br>
DOI: <a href="https://doi.org/10.11578/rubin/dc.20250909.20">10.11578/rubin/dc.20250909.20</a> <br>

**Learning objective:** To understand and access the calibration images (bias, dark, and flat frames).

**LSST data products:** `bias`, `dark`, and `flat`

**Packages:** `lsst.daf.butler`, `lsst.afw.display`

**Credit:**
Originally developed by the Rubin Community Science team.
Please consider acknowledging them if this notebook is used for the preparation of journal articles, software releases, or other notebooks.

**Get Support:**
Everyone is encouraged to ask questions or raise issues in the 
<a href="https://community.lsst.org/c/support">Support Category</a> 
of the Rubin Community Forum.
Rubin staff will respond to all questions posted there.

## 1. Introduction

The process of Instrument Signature Removal (ISR; also called "image reduction") uses bias, dark, and flat field calibration frames as part of the process to transform raw images into visit images.
ISR is the first step of image processing, which removes instrumental effects introduced in the raw images by the telescope and detectors and produces an accurate representation of the incoming light.
For descriptions of the ISR steps and the generation, verification, certification, approval, and distribution of the calibration products necessary for ISR, refer to
[Plazas Malag\u00f3n et al. (2025)](https://ui.adsabs.harvard.edu/abs/2025JATIS..11a1209P/abstract),
the "Verifying LSST Calibration Data Products" technical note ([DMTN-101](https://dmtn-101.lsst.io/)),
and the "Calibration Generation, Verification, Acceptance, and Certification" technical note ([DMTN-222](https://dmtn-222.lsst.io/)).

**Bias images:** An exposure obtained with zero exposure time to measure the pedestal level of counts applied during readout.

**Dark frames:** An exposure obtained with a nonzero exposure time but with no illumination (shutter closed) to measure the detector's response to the thermal energy in the camera.

**Flat fields:** An exposure taken with even illumination across the field to measure pixel response variations.

**Related tutorials:** See the 200-level tutorial on visit images, and the 100-level tutorials on how to use the Butler.

### 1.1. Import packages

Import the Rubin data `Butler` from the `lsst.daf.butler` package
([pipelines.lsst.io](https://pipelines.lsst.io)) and the `afw.display` package to retrieve and display calibration images.
Import `numpy` ([numpy.org](https://numpy.org)), a fundamental package for scientific computing with arrays in Python.

In [ ]:
from lsst.daf.butler import Butler
import lsst.afw.display as afwDisplay
import numpy as np

### 1.2. Define parameters and functions

Instantiate the Butler for the DP2 data release.

In [ ]:
butler = Butler('dp2_prep',
                collections=['LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2'])
assert butler is not None

Set the display backend to Firefly.

In [ ]:
afwDisplay.setDefaultBackend('firefly')

Define the visit and detector identifiers to obtain calibration frames for.

In [ ]:
my_visit = 2025071200709
my_detector = 10

## 2. Data access

The calibration frames are only accessible via the Butler.

Show the Butler dimensions for each calibration frame type.
Notice that only the `flat` frames are by band (by filter).

In [ ]:
for frame_type in ['raw', 'bias', 'dark', 'flat']:
    print(' ')
    print(frame_type)
    print(butler.get_dataset_type(frame_type))
    print('Required dimensions: ',
          butler.get_dataset_type(frame_type).dimensions.required)

### 2.1. Retrieve and display frames

Retrieve the raw exposure, and the bias, dark, and flat frames, all for the same
visit and display them in separate frames with Firefly.

In [ ]:
raw = butler.get("raw", exposure=my_visit, detector=my_detector)
afw_display = afwDisplay.Display(frame=1)
afw_display.image(raw.image)

In [ ]:
bias = butler.get("bias", visit=my_visit, detector=my_detector)
afw_display = afwDisplay.Display(frame=2)
afw_display.image(bias)

In [ ]:
dark = butler.get("dark", visit=my_visit, detector=my_detector)
afw_display = afwDisplay.Display(frame=3)
afw_display.image(dark)

In [ ]:
flat = butler.get("flat", visit=my_visit, detector=my_detector)
afw_display = afwDisplay.Display(frame=4)
afw_display.image(flat)

## 3. Pixel data

The bias, dark, and flat frames have their image plane and a variance plane.

There is also a mask plane but it is unpopulated. The mask plane of the `raw` image is used in processing.

### 3.1. Image plane

Pixel data in units of ADU (analog-digital units).

Print the minimum, maximum, mean, and standard deviation of the pixel values from the image planes of the bias, dark, and flat frames.

In [ ]:
print('image     min       max            mean     std     ')
print('----------------------------------------------------')
print(f"bias  {np.min(bias.image.array):7.2f}  {np.max(bias.image.array):9.2f} \
       {np.mean(bias.image.array):7.2f}  {np.std(bias.image.array):7.2f}")
print(f"dark  {np.min(dark.image.array):7.2f}  {np.max(dark.image.array):9.2f} \
       {np.mean(dark.image.array):7.2f}  {np.std(dark.image.array):7.2f}")
print(f"flat  {np.min(flat.image.array):7.2f}  {np.max(flat.image.array):9.2f} \
       {np.mean(flat.image.array):7.2f}  {np.std(flat.image.array):7.2f}")

### 3.2. Variance plane

Pixel data in units of ADU^2 (analog-digital units squared).

Print the minimum, mean, and maximum pixel value from the variance planes of the bias, dark, and flat frames.

In [ ]:
print('image    min            mean        max  ')
print('-----------------------------------------')
print(f"bias  {np.min(bias.variance.array):7.2f} \
       {np.mean(bias.variance.array):7.2f}  {np.max(bias.variance.array):11.2f}")
print(f"dark  {np.min(dark.variance.array):7.2f} \
       {np.mean(dark.variance.array):7.2f}  {np.max(dark.variance.array):11.2f}")
print(f"flat  {np.min(flat.variance.array):7.2f} \
       {np.mean(flat.variance.array):7.2f}  {np.max(flat.variance.array):11.2f}")

### 3.3. No mask frame

Show that the mask frame is unpopulated (all zero values).

In [ ]:
print('bias ', np.min(bias.mask.array),
      np.mean(bias.mask.array), np.max(bias.mask.array))
print('dark ', np.min(dark.mask.array),
      np.mean(dark.mask.array), np.max(dark.mask.array))
print('flat ', np.min(flat.mask.array),
      np.mean(flat.mask.array), np.max(flat.mask.array))

## 4. Metadata

The bias, dark, and flat frames are of type `ExposureF` and come with header information.

### 4.1. Header

The informational equivalent of a FITS image header is the `metadata`.

Get the metadata for the bias, dark, or flat.

In [ ]:
metadata = bias.getMetadata()
# metadata = dark.getMetadata()
# metadata = flat.getMetadata()

Option to display the long list of metadata.

In [ ]:
# metadata

Convert the metadata to a python dictionary.

In [ ]:
md_dict = metadata.toDict()

Show any metadata key that contains the string 'UNIT' or 'MJD'.

In [ ]:
temp = 'UNIT'
# temp = 'MJD'
for key in md_dict.keys():
    if key.find(temp) >= 0:
        print(key)

Print the 'BUNIT'.

In [ ]:
md_dict['BUNIT']

Clean up.

In [ ]:
del md_dict, metadata

### 4.2. Bounding box 

The bounding box defines the extent (corners) of the image.

Get the bounding box; use the bias frame for this example.

In [ ]:
bbox = bias.getBBox()

In [ ]:
bbox

Print the start and end pixels in the X and Y dimensions.

In [ ]:
print(bbox.beginX, bbox.beginY)
print(bbox.endX, bbox.endY)

Clean up.

In [ ]:
del bbox